In [3]:
%pip install admet-ai
import csv
import os
import sys
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, QED, RDConfig
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem import Lipinski
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.Crippen import MolLogP
from admet_ai import ADMETModel
import sascorer

### Disable RDKit informational messages ###
RDLogger.DisableLog('rdApp.*')

#############
### Setup ###
#############

### Load SMILES ###
supply = Chem.SmilesMolSupplier(
    'standardized_smiles.smi',
    delimiter='\t',
    titleLine=False
    )

### ADMET ###
model = ADMETModel(include_physchem=False)
admet_endpoints = [
            "hERG",
            "Caco2_Wang",
            "BBB_Martins",
            "Clearance_Hepatocyte_AZ",
            ]

### PAINS, Brenk, NIH ###

# PAINS flag
params_pains = FilterCatalogParams()
params_pains.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
catalog_pains = FilterCatalog(params_pains)

# Brenk Flag
params_unwanted = FilterCatalogParams()
params_unwanted.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
catalog_unwanted = FilterCatalog(params_unwanted)

# NIH Flag
params_nih = FilterCatalogParams()
params_nih.AddCatalog(FilterCatalogParams.FilterCatalogs.NIH)
catalog_nih = FilterCatalog(params_nih)

### Synthesizability score ###
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))


###############################
### Filtering and Profiling ###
###############################

def calculate_pains_brenk(mol):

    # Check for PAINS
    pains_match = catalog_pains.HasMatch(mol)

    # Check for BRENK
    brenk_match = catalog_unwanted.HasMatch(mol)

    # Check for NIH
    NIH_match = catalog_nih.HasMatch(mol)
    
    return pains_match, brenk_match, NIH_match


def calculate_profile(mol, admet_model=None):

    smiles = Chem.MolToSmiles(mol)

    # -----------------------------------------
    # 1. PAINS / BRENK
    # -----------------------------------------

    pains_match, brenk_match, NIH_match = calculate_pains_brenk(mol)

    # -----------------------------------------
    # 2. RDKit descriptors
    # -----------------------------------------

    profile = {
        "Compound": smiles,
        "MW": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Crippen.MolLogP(mol), 2),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "Aromatic_Ring_Number": rdMolDescriptors.CalcNumAromaticRings(mol),
        "TPSA": round(Descriptors.TPSA(mol), 2),
        "RotB": Descriptors.NumRotatableBonds(mol),
        "QED": round(QED.qed(mol), 2),
        "Formal_Charge": Chem.GetFormalCharge(mol),
        "SA_score": round(sascorer.calculateScore(mol),2),
        "PAINS": pains_match,
        "BRENK": brenk_match,
        "NIH": NIH_match
    }

    # -----------------------------------------
    # 3. ADMET
    # -----------------------------------------

    if admet_model is not None:

        preds = admet_model.predict(smiles=smiles)

        for endpoint in admet_endpoints:

            value = preds.get(endpoint)

            profile[endpoint] = (
                round(value, 2)
                if value is not None
                else None
            )

    else:

        for endpoint in admet_endpoints:
            profile[endpoint] = None

    return profile # Returns a dictionary containing the above defined ADMET parameters

profile_table = []

# Iterate through all compounds in smi file
for i, mol in enumerate(supply, start=0):
    if mol is not None:
        profile_table.append(calculate_profile(mol, admet_model=model))
        
##############
### Output ###
##############

### Save results as csv file ###
df = pd.DataFrame(profile_table)
saved_path = 'standardized_smiles_admet.csv'
df.to_csv(saved_path, index=False)

print(f'The results have been successfully saved as {saved_path}!') 

Note: you may need to restart the kernel to use updated packages.


SMILES to Mol: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1123.57it/s]
Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 1, batch_size = 64)
GPU available: False, used: False                                                                | 0/2 [00:00<?, ?it/s]
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

C:\Users\pham.congdat\AppData\Local\anaconda3\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\pham.congdat\AppData\Local\anaconda3\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Output()

...

Output()

Output()

Output()

model ensembles: 100%|███████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  2.67it/s]

The results have been successfully saved as standardized_smiles_admet.csv!
